## Cargar datos y librerias

## Librerias

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

## Datos

In [0]:
df = spark.read.csv("/Volumes/prueba/ai_data/dataset_trabajosai/ai_job_market.csv", header=True, inferSchema=True)
display(df)

In [0]:
df = df.withColumn("salary_lower_bound", split(col("salary_range_usd"), "-")[0].cast("double")) \
       .withColumn("salary_upper_bound", split(col("salary_range_usd"), "-")[1].cast("double")) \
       .withColumn("salary_avg", (col("salary_lower_bound") + col("salary_upper_bound")) / 2)

display(df)

In [0]:
display(df.select("salary_avg"))

##  Salario promedio por nivel de experiencia

In [0]:
df_grouped = df.groupBy("experience_level").agg(avg("salary_avg").alias("avg_salary"))
display(df_grouped)

## Corrlacion entre salario y tipoo de contrato

In [0]:
df_grouped_employment = df.groupBy("employment_type").agg(avg("salary_avg").alias("avg_salary"))
display(df_grouped_employment)

## Comparación por tipo de trabajo

In [0]:
df_work_type = df.groupBy("employment_type").count()
display(df_work_type)

In [0]:
df_job_title = df.groupBy("job_title").count()
display(df_job_title)

## Salarios promedios por lugar

In [0]:
df_location_salary = df.groupBy("location").agg(avg("salary_avg").alias("avg_salary")).orderBy(col("avg_salary").desc()).limit(10)
display(df_location_salary)

### Numero de ofertas por mes

In [0]:
jobs_by_month = (
    df.withColumn("month", date_trunc("month", col("posted_date")))
      .groupBy("month")
      .agg(count("*").alias("num_offers"))
      .orderBy("month")
)

display(jobs_by_month)

## Ofertas por tiempo de data scientist

In [0]:
salary_evolution_ds = (
    df.select("posted_date", "job_title", "salary_avg")
      .filter(lower(col("job_title")).like("%data scientist%"))
      .withColumn("year", year(col("posted_date")))
      .groupBy("year")
      .agg(avg("salary_avg").alias("avg_salary"))
      .orderBy("year")
)

display(salary_evolution_ds)

In [0]:
salary_evolution_ds = (
    df.select("posted_date", "job_title", "salary_avg")
      .filter(lower(col("job_title")).like("%data scientist%"))
      .withColumn("month", date_trunc("month", col("posted_date")))
      .groupBy("month")
      .agg(avg("salary_avg").alias("avg_salary"))
      .orderBy("month")
)

display(salary_evolution_ds)

## Herramientas

In [0]:

tools_df = (
    df.select(explode(split(col("tools_preferred"), ",")).alias("tool"))
      .withColumn("tool", trim(lower(col("tool"))))
      .groupBy("tool")
      .count()
      .orderBy(col("count").desc())
      .limit(10)
)

display(tools_df)